# EEG_07e — Pre-costruzione Tensori Grafo / Ipergrafo da CSV

Legge i CSV di Paolo (`data/raw_csv/training_set/PXXX_SYYY/parola_img.csv`)
e costruisce tensori PyG `Data(x, edge_index, y, subj, sess)` in `data/interim/graphs/`.

**Struttura CSV**: 61 righe × 384 colonne (canali × campioni), nessun header.

Supporta:
- Metodi connettività: **PCC**, **PLV**, **wPLI**
- K valori per k-NN
- **Pruning**: soglia edge + rimozione canali a bassa connettività
- Costruzione **ipergrafo** (per EEG_11)

In [ ]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

# Metodi connettività
METHODS = ["pcc"]           # "pcc" | "plv" | "wpli"

# K-vicini per k-NN graph
K_VALUES = [6]

# Soglia edge: rimuove archi con peso < threshold (0.0 = nessuna)
EDGE_THRESHOLD = 0.0

# Pruning canali: rimuove canali con connettività media < mean - sigma*std
# None = nessun pruning
CHANNEL_PRUNING_SIGMA = None

# Canali da escludere esplicitamente (es. ["A1", "A2"] o [])
CHANNEL_DROP_NAMES = []

# Costruisci anche ipergrafo (per EEG_11)
BUILD_HGNN = True
K_HYPER    = 6

# Schema label da applicare
CLUSTER_SCHEME = "concr4"   # "concr4" | "ward4" | "sem5" | "pos4" | "raw110"

# Forza ricostruzione anche se file già esiste
FORCE_REBUILD = True

print("Config OK")

In [ ]:
# ============================================================
# IMPORT E PATHS
# ============================================================

import os, sys, json
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import torch

torch.multiprocessing.set_sharing_strategy('file_system')  # evita OOM su /dev/shm condivisa
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from torch_geometric.data import Data

# Trova project root
project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

# Paths
CSV_ROOT   = project_root / "data" / "raw_csv" / "training_set"
GRAPHS_DIR = project_root / "data" / "interim" / "graphs"
CONFIGS    = project_root / "configs" / "label_schemes"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

N_CHANS   = 61
N_SAMPLES = 384

# Mapping parola → label_id (da label2idx.json)
with open(CONFIGS / "label2idx.json") as f:
    word2labelid = json.load(f)   # {"accendere": 0, "acqua": 1, ...}

# Schema cluster
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(CLUSTER_SCHEME, project_root / "data" / "interim")

# Lista cartelle soggetto-sessione
session_dirs = sorted(CSV_ROOT.iterdir())
print(f"Project root  : {project_root}")
print(f"CSV root      : {CSV_ROOT}")
print(f"Cartelle sess : {len(session_dirs)}")  # ~357
print(f"N_CHANS       : {N_CHANS}")
print(f"Schema        : {CLUSTER_SCHEME} ({N_CLASSES} classi)")
print(f"Esempio dir   : {session_dirs[0].name}")

In [ ]:
# ============================================================
# PARSING NOMI CARTELLA
# PXXX_SYYY → subj_id (int), sess_id (int)
# ============================================================

def parse_folder(folder_name: str):
    """Es. 'P003_S002' → (3, 2)"""
    parts = folder_name.split("_")
    subj = int(parts[0][1:])   # rimuove 'P'
    sess = int(parts[1][1:])   # rimuove 'S'
    return subj, sess

def load_csv_trial(csv_path: Path) -> np.ndarray:
    """Legge CSV 61×384 senza header. Restituisce array float32 (61, 384)."""
    return pd.read_csv(csv_path, header=None).values.astype(np.float32)

# Test
test_dir = session_dirs[0]
test_csv = sorted(test_dir.iterdir())[0]
x_test   = load_csv_trial(test_csv)
subj_t, sess_t = parse_folder(test_dir.name)
word_t   = test_csv.stem.replace("_img", "")

print(f"Cartella: {test_dir.name} → subj={subj_t}, sess={sess_t}")
print(f"File    : {test_csv.name} → parola='{word_t}', label_id={word2labelid.get(word_t, '??')}")
print(f"Shape   : {x_test.shape}  (atteso: (61, 384))")
assert x_test.shape == (N_CHANS, N_SAMPLES), f"Shape attesa ({N_CHANS}, {N_SAMPLES}), trovata {x_test.shape}"
print("✅ Parsing OK")

In [ ]:
# ============================================================
# FUNZIONI DI CONNETTIVITÀ
# ============================================================

def pcc_matrix(x_np: np.ndarray) -> np.ndarray:
    """Pearson |PCC| tra canali. Shape: (N, N)"""
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return pcc


def plv_matrix(x_np: np.ndarray) -> np.ndarray:
    """Phase Locking Value via Hilbert. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    phases = np.angle(hilbert(x_np, axis=1))
    plv = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            diff = phases[i] - phases[j]
            plv[i, j] = plv[j, i] = np.abs(np.mean(np.exp(1j * diff)))
    return plv


def wpli_matrix(x_np: np.ndarray) -> np.ndarray:
    """Weighted Phase Lag Index. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    analytic = hilbert(x_np, axis=1)
    wpli = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            cs = analytic[i] * np.conj(analytic[j])
            im = np.imag(cs)
            w  = np.abs(im)
            wpli[i, j] = wpli[j, i] = np.abs(np.mean(im * w)) / (np.mean(w) + 1e-9)
    return wpli


def knn_edge_index(matrix: np.ndarray, k: int,
                   threshold: float = 0.0) -> torch.LongTensor:
    """k-NN graph da matrice connettività. Restituisce edge_index (2, E)."""
    N = matrix.shape[0]
    rows, cols = [], []
    for i in range(N):
        row = matrix[i].copy(); row[i] = -1.0
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k = np.argsort(row)[-k:]
        for j in top_k:
            if row[j] > 0.0 or threshold == 0.0:
                rows += [i, j]; cols += [j, i]
    return torch.tensor([rows, cols], dtype=torch.long)


def hyperedge_index_fn(x_np: np.ndarray, k: int,
                       threshold: float = 0.0) -> torch.LongTensor:
    """
    Ipergrafo k-NN per-trial basato su PCC.
    Ogni nodo i è centro di un'iperedge con i + top-k vicini.
    """
    pcc = pcc_matrix(x_np)
    N   = pcc.shape[0]
    vertex_list, edge_list = [], []
    for e_id in range(N):
        row = pcc[e_id].copy()
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k   = np.argsort(row)[-k:]
        members = [e_id] + [j for j in top_k if (row[j] > 0.0 or threshold == 0.0)]
        for v in members:
            vertex_list.append(v); edge_list.append(e_id)
    return torch.tensor([vertex_list, edge_list], dtype=torch.long)


print("Funzioni connettività OK")

In [ ]:
# ============================================================
# ANALISI VISIVA — connettività per canale
# Aiuta a decidere se/cosa prunare
# ============================================================

# Usa primo soggetto come campione
sample_dir = session_dirs[0]
csv_files  = sorted(sample_dir.iterdir())[:50]   # 50 trial campione

print(f"Calcolo PCC media su {len(csv_files)} trial ({sample_dir.name})...")
pcc_sum = np.zeros((N_CHANS, N_CHANS))
for csv_path in tqdm(csv_files):
    x_np = load_csv_trial(csv_path)
    pcc_sum += pcc_matrix(x_np)
pcc_avg = pcc_sum / len(csv_files)

ch_connectivity = pcc_avg.sum(axis=1)
mean_conn = ch_connectivity.mean()
std_conn  = ch_connectivity.std()

# Nomi canali dall'ordine nel CSV (basato su ebneuro.csv se disponibile)
ch_labels = [str(i) for i in range(N_CHANS)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(pcc_avg, ax=axes[0], cmap="RdYlBu_r", vmin=0, vmax=0.5)
axes[0].set_title(f"PCC Media — {sample_dir.name} ({len(csv_files)} trial)")

colors = ["red" if c < mean_conn - 2*std_conn else "steelblue" for c in ch_connectivity]
axes[1].bar(range(N_CHANS), ch_connectivity, color=colors)
axes[1].axhline(mean_conn, color="black", linestyle="--", label="Media")
axes[1].axhline(mean_conn - std_conn, color="orange", linestyle="--", label="Mean-1σ")
axes[1].axhline(mean_conn - 2*std_conn, color="red", linestyle="--", label="Mean-2σ")
axes[1].set_xlabel("Canale (indice)")
axes[1].set_title("Connettività per Canale")
axes[1].legend()

plt.tight_layout()
fig_path = project_root / "figures" / f"eeg07e_connectivity_{sample_dir.name}.png"
fig.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()

print("\nCanali sotto mean-2σ (candidati pruning):")
for i, conn in enumerate(ch_connectivity):
    if conn < mean_conn - 2*std_conn:
        print(f"  idx={i:2d}  conn={conn:.4f}")

In [ ]:
# ============================================================
# PRUNING CANALI
# ============================================================

# Indici attivi (tutti e 61 di default)
active_idx = list(range(N_CHANS))

# Pruning per sigma
if CHANNEL_PRUNING_SIGMA is not None:
    thresh = mean_conn - CHANNEL_PRUNING_SIGMA * std_conn
    active_idx = [i for i in active_idx if ch_connectivity[i] >= thresh]
    print(f"Pruning σ={CHANNEL_PRUNING_SIGMA}: rimasti {len(active_idx)}/{N_CHANS} canali")

# Pruning per nome (richiede mapping idx→nome se disponibile)
if CHANNEL_DROP_NAMES:
    print(f"Drop esplicito: {CHANNEL_DROP_NAMES} — imposta indici in CHANNEL_DROP_NAMES come indici numerici")

N_ACTIVE = len(active_idx)
print(f"Canali attivi: {N_ACTIVE}/{N_CHANS}")

In [ ]:
# ============================================================
# FUNZIONE HELPER: itera tutti i trial da CSV
# ============================================================

def iter_all_trials():
    """
    Generator: itera tutte le cartelle PXXX_SYYY e tutti i CSV.
    Yields: (x_np, label_id, cluster_id, subj_id, sess_id)
    Salta file con parola non in label2idx o label_id non in labelid2cluster.
    """
    for sess_dir in sorted(CSV_ROOT.iterdir()):
        if not sess_dir.is_dir():
            continue
        subj_id, sess_id = parse_folder(sess_dir.name)
        for csv_path in sorted(sess_dir.iterdir()):
            if csv_path.suffix != ".csv":
                continue
            word = csv_path.stem.replace("_img", "")
            if word not in word2labelid:
                continue
            label_id = word2labelid[word]
            if label_id not in labelid2cluster:
                continue
            cluster_id = labelid2cluster[label_id]
            x_np = load_csv_trial(csv_path)[active_idx, :]   # (N_ACTIVE, 384)
            yield x_np, label_id, cluster_id, subj_id, sess_id

# Conta trial totali
n_total = sum(1 for _ in iter_all_trials())
print(f"Trial totali validi: {n_total}")
print(f"(atteso ~{len(session_dirs) * 110} = {len(session_dirs)} sess × 110 parole)")

In [ ]:
# ============================================================
# BUILD GRAPH TENSORS (PCC / PLV / wPLI)
# Output: data/interim/graphs/graph_{method}_k{k}.pt
# ============================================================

CONN_FN = {"pcc": pcc_matrix, "plv": plv_matrix, "wpli": wpli_matrix}

for method in METHODS:
    for k in K_VALUES:
        out_path = GRAPHS_DIR / f"graph_{method}_k{k}.pt"
        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip: {out_path.name}")
            continue

        print(f"\nCostruendo {out_path.name}...")
        conn_fn   = CONN_FN[method]
        data_list = []

        for x_np, label_id, cluster_id, subj_id, sess_id in tqdm(
                iter_all_trials(), total=n_total, desc=f"{method} k={k}"):

            matrix     = conn_fn(x_np)
            edge_index = knn_edge_index(matrix, k=k, threshold=EDGE_THRESHOLD)

            data_list.append(Data(
                x          = torch.tensor(x_np, dtype=torch.float32),
                edge_index = edge_index,
                y          = torch.tensor(cluster_id, dtype=torch.long),
                label_id   = torch.tensor(label_id,   dtype=torch.long),
                subj       = torch.tensor(subj_id,    dtype=torch.long),
                sess       = torch.tensor(sess_id,    dtype=torch.long),
            ))

        torch.save(data_list, out_path)
        print(f"  ✅ Salvato: {out_path.name} — {len(data_list)} grafi  x.shape={data_list[0].x.shape}")

In [ ]:
# ============================================================
# BUILD HYPERGRAPH TENSORS
# Output: data/interim/graphs/hgraph_k{k}.pt
# ============================================================

if BUILD_HGNN:
    for k in [K_HYPER]:
        out_path = GRAPHS_DIR / f"hgraph_k{k}.pt"
        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip: {out_path.name}")
            continue

        print(f"\nCostruendo {out_path.name}...")
        data_list = []

        for x_np, label_id, cluster_id, subj_id, sess_id in tqdm(
                iter_all_trials(), total=n_total, desc=f"hgraph k={k}"):

            he_index     = hyperedge_index_fn(x_np, k=k, threshold=EDGE_THRESHOLD)
            n_hyperedges = he_index[1].max().item() + 1

            data_list.append(Data(
                x               = torch.tensor(x_np, dtype=torch.float32),
                hyperedge_index = he_index,
                num_hyperedges  = n_hyperedges,
                y               = torch.tensor(cluster_id, dtype=torch.long),
                label_id        = torch.tensor(label_id,   dtype=torch.long),
                subj            = torch.tensor(subj_id,    dtype=torch.long),
                sess            = torch.tensor(sess_id,    dtype=torch.long),
            ))

        torch.save(data_list, out_path)
        print(f"  ✅ Salvato: {out_path.name} — {len(data_list)} ipergrafi")

In [ ]:
# ============================================================
# SANITY CHECK
# ============================================================

print(f"=== File in {GRAPHS_DIR} ===")
for pt_file in sorted(GRAPHS_DIR.glob("*.pt")):
    dl = torch.load(pt_file, weights_only=False)
    d0 = dl[0]
    subj_ids = sorted(set(d.subj.item() for d in dl))
    if hasattr(d0, "edge_index"):
        print(f"  {pt_file.name}: {len(dl)} grafi | "
              f"x={d0.x.shape} | ei={d0.edge_index.shape} | "
              f"n_subj={len(subj_ids)} | y_range=[{min(d.y.item() for d in dl)},{max(d.y.item() for d in dl)}]")
    else:
        print(f"  {pt_file.name}: {len(dl)} ipergrafi | "
              f"x={d0.x.shape} | he={d0.hyperedge_index.shape} | n_he={d0.num_hyperedges}")

print(f"\n✅ Sanity check OK")
print(f"   Canali: {N_ACTIVE}/{N_CHANS} {'(pruned)' if N_ACTIVE < N_CHANS else '(tutti 61)'}")
print(f"   Schema : {CLUSTER_SCHEME} ({N_CLASSES} classi: {cluster_names})")